# Your first SAE in 30 min - Gemma-2-2B on Colab Free

Welcome! A **Sparse Autoencoder (SAE)** is a tiny neural net that takes the rich, tangled activations of a language model and rewrites them as a long list of *almost-zero-except-for-a-few* features. The handful that fire end up being human-readable concepts - "this text mentions Paris", "this is Python code", "sarcasm detected" - so SAEs are the workhorse of mechanistic interpretability.

By the end of this notebook you'll have **your own trained SAE for Gemma-2-2B layer 15 uploaded to your HuggingFace account**, a working grasp of the TopK training loop, and everything you need to graduate to the larger Kaggle / paper-grade ladder. Gemma-2-2B in bf16 fits comfortably in a T4's 16 GB with room for activation capture, and 50M tokens at width 16384 lands in roughly 30-40 minutes. Let's go.

In [ ]:
!pip install -q -U transformers accelerate datasets safetensors einops huggingface_hub
!nvidia-smi | head -10

## Configuration - tweak only if you know what you're doing

All the knobs live in one cell so you never have to hunt for them. The defaults are tuned for a free Colab T4 (16 GB). The single thing you **must** change is `HF_USERNAME`.

In [ ]:
import torch

MODEL_ID       = 'google/gemma-2-2b'
LAYER          = 15                  # mid-layer residual
D_MODEL        = 2304
N_FEATURES     = 16384               # 7x expansion (T4-friendly)
K_TOPK         = 64
K_AUX          = 256
ALPHA_AUX      = 1/32
DEAD_TOKENS    = 5_000_000
TOKEN_BUDGET   = 50_000_000          # ~30 min on T4
SEQ_LEN        = 512
FWD_BATCH      = 4
BATCH_SIZE     = 2048                # SAE training batch
LR_PEAK        = 2e-4
LR_FLOOR       = 6e-5
WARMUP_STEPS   = 1000
CKPT_EVERY_TOK = 5_000_000
HF_USERNAME    = 'YOUR_HF_USERNAME'   # <-- USER EDITS THIS
HF_REPO        = f'{HF_USERNAME}/gemma2-2b-sae-first'
DRIVE_DIR      = '/content/drive/MyDrive/openinterp_ckpt'

DEVICE = 'cuda'
print(f'Training a {N_FEATURES}-feature TopK-{K_TOPK} SAE on {MODEL_ID} layer {LAYER}')
print(f'Budget: {TOKEN_BUDGET:,} tokens -> ~{TOKEN_BUDGET // (FWD_BATCH * SEQ_LEN):,} forward passes')

## Auth: mount Drive + set HF token via Colab Secrets

We mount Google Drive so checkpoints survive the 90-minute Colab idle-disconnect. Add a secret named **`HF_TOKEN`** in the left sidebar (key icon), paste a write-enabled HF token, and toggle "Notebook access".

In [ ]:
import os
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)
print(f'Drive checkpoint dir: {DRIVE_DIR}')
print(f'HF auth OK - will push to: {HF_REPO}')

## Load Gemma-2-2B in bf16

The base model is frozen - we only train the SAE. bf16 + SDPA attention fits the 2B params plus activations on a T4.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,           # NOTE: `dtype=`, not `torch_dtype=` (transformers 4.57+)
    device_map='cuda',
    attn_implementation='sdpa',     # SDPA works on T4; flash-attn does not
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

vram_gb = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded. VRAM used: {vram_gb:.2f} GB / 16 GB')
print(f'Residual stream dim at layer {LAYER}: {D_MODEL}')

## Hook the residual stream + stream training tokens

We attach a forward hook to layer 15 that grabs the post-block residual activations. Tokens come from FineWeb-Edu (streamed - no download). We pack sequences to exactly `SEQ_LEN` so every batch gives `FWD_BATCH * SEQ_LEN` activations.

In [ ]:
from datasets import load_dataset

# The hook captures the layer's output tuple; out[0] is the residual tensor
_act_cache = {}
def _hook(module, inp, out):
    _act_cache['x'] = out[0] if isinstance(out, tuple) else out

hook_handle = model.model.layers[LAYER].register_forward_hook(_hook)

# Streaming corpus - no download needed
corpus = load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT',
                      split='train', streaming=True)

def token_packer(stream, seq_len=SEQ_LEN, batch=FWD_BATCH):
    """Yield (batch, seq_len) int64 token tensors, packing docs end-to-end."""
    buf = []
    for doc in stream:
        ids = tok.encode(doc['text'], add_special_tokens=False)
        buf.extend(ids + [tok.eos_token_id])
        while len(buf) >= batch * seq_len:
            chunk = buf[:batch * seq_len]
            buf = buf[batch * seq_len:]
            yield torch.tensor(chunk, dtype=torch.long).view(batch, seq_len)

def activation_stream(token_iter):
    """Yield (B*T, D_MODEL) bf16 activation tensors on GPU."""
    for ids in token_iter:
        ids = ids.to(DEVICE, non_blocking=True)
        with torch.no_grad():
            model(ids, use_cache=False)
        x = _act_cache['x'].reshape(-1, D_MODEL)  # (B*T, D)
        yield x.float()                            # train SAE in fp32 for stability

print('Activation stream primed.')

## TopK SAE with AuxK (Gao et al. 2024)

**TopK**: pick the largest `K` pre-activations per token, zero the rest. **AuxK**: every so often, ask "dead" features (those that haven't fired in a while) to reconstruct the residual error using the next `K_AUX` largest - this resurrects them. We initialise `b_dec` as the geometric median of a data sample (Weiszfeld iteration), which makes the decoder start near the data cloud and trains much faster than zero-init.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def geometric_median(X, n_iter=25, eps=1e-6):
    # Weiszfeld: median minimises sum of L2 distances; great init for b_dec
    y = X.mean(0)
    for _ in range(n_iter):
        d = (X - y).norm(dim=1).clamp_min(eps)
        y = (X / d[:, None]).sum(0) / (1.0 / d).sum()
    return y

class TopKSAE(nn.Module):
    def __init__(self, d_in, d_sae, k, k_aux):
        super().__init__()
        self.d_in, self.d_sae, self.k, self.k_aux = d_in, d_sae, k, k_aux
        # Tied init: decoder unit-normed, encoder = decoder.T
        W = torch.randn(d_in, d_sae)
        W /= W.norm(dim=0, keepdim=True)
        self.W_dec = nn.Parameter(W.T.contiguous())          # (d_sae, d_in)
        self.W_enc = nn.Parameter(W.contiguous())            # (d_in, d_sae)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_in))
        self.register_buffer('last_fired', torch.zeros(d_sae, dtype=torch.long))
        self._step = 0

    @torch.no_grad()
    def init_b_dec(self, X):
        self.b_dec.data = geometric_median(X).to(self.b_dec)

    def encode_pre(self, x):
        return (x - self.b_dec) @ self.W_enc + self.b_enc    # (N, d_sae)

    @staticmethod
    def _topk_scatter(pre, k):
        # Keep top-k per row; zero everything else. Non-negative via ReLU-after-mask.
        vals, idx = pre.topk(k, dim=-1)
        out = torch.zeros_like(pre)
        out.scatter_(-1, idx, F.relu(vals))
        return out, idx

    def forward(self, x, dead_mask=None):
        pre = self.encode_pre(x)
        z, idx = self._topk_scatter(pre, self.k)
        x_hat = z @ self.W_dec + self.b_dec
        err = x - x_hat

        aux_loss = x.new_zeros(())
        if dead_mask is not None and dead_mask.any():
            # AuxK: let dead features reconstruct the residual error
            pre_dead = pre.masked_fill(~dead_mask, float('-inf'))
            k_eff = min(self.k_aux, int(dead_mask.sum().item()))
            if k_eff > 0:
                z_aux, _ = self._topk_scatter(pre_dead, k_eff)
                err_hat = z_aux @ self.W_dec
                aux_loss = (err - err_hat).pow(2).mean()

        recon_loss = err.pow(2).mean()
        # Track firing for dead-feature detection
        fired = torch.zeros(self.d_sae, device=x.device, dtype=torch.bool)
        fired.scatter_(0, idx.reshape(-1), True)
        self.last_fired[fired] = self._step
        return x_hat, recon_loss, aux_loss, z

    @torch.no_grad()
    def renorm_decoder(self):
        # Keep decoder columns unit-norm (standard SAE stability trick)
        self.W_dec.data /= self.W_dec.data.norm(dim=1, keepdim=True).clamp_min(1e-8)

print('TopKSAE defined.')

## Training loop - saves to Drive every 5M tokens, uploads to HF at end

This is the main event. We stream activations, train the SAE, and checkpoint to Drive so a Colab disconnect only costs you a few minutes. If a checkpoint already exists in Drive we resume automatically.

In [ ]:
import json, math, time
from pathlib import Path
from tqdm.auto import tqdm
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo

sae = TopKSAE(D_MODEL, N_FEATURES, K_TOPK, K_AUX).to(DEVICE)
opt = torch.optim.Adam(sae.parameters(), lr=LR_PEAK, betas=(0.9, 0.999))

ckpt_path = Path(DRIVE_DIR) / 'sae_latest.pt'
tokens_seen = 0
step = 0

# --- Resume from Drive if a checkpoint exists (idle-disconnect recovery) ---
if ckpt_path.exists():
    ck = torch.load(ckpt_path, map_location=DEVICE)
    sae.load_state_dict(ck['sae'])
    opt.load_state_dict(ck['opt'])
    tokens_seen = ck['tokens_seen']
    step = ck['step']
    print(f'Resumed from {ckpt_path} @ {tokens_seen:,} tokens / step {step}')
else:
    print('Fresh run - will init b_dec from first batch.')

def lr_at(step):
    # Linear warmup -> cosine decay from peak to floor
    if step < WARMUP_STEPS:
        return LR_PEAK * step / max(1, WARMUP_STEPS)
    total = TOKEN_BUDGET // BATCH_SIZE
    p = (step - WARMUP_STEPS) / max(1, total - WARMUP_STEPS)
    p = min(1.0, max(0.0, p))
    return LR_FLOOR + 0.5 * (LR_PEAK - LR_FLOOR) * (1 + math.cos(math.pi * p))

# Activation buffer so we can draw BATCH_SIZE rows at a time
buf = torch.empty(0, D_MODEL, device=DEVICE)
act_iter = activation_stream(token_packer(corpus))
last_ckpt_tok = tokens_seen
pbar = tqdm(total=TOKEN_BUDGET, initial=tokens_seen, unit='tok', unit_scale=True)
t0 = time.time()

while tokens_seen < TOKEN_BUDGET:
    # Fill buffer until we have at least one SAE batch worth of activations
    while buf.shape[0] < BATCH_SIZE:
        x_new = next(act_iter)
        buf = torch.cat([buf, x_new], dim=0)

    x = buf[:BATCH_SIZE]
    buf = buf[BATCH_SIZE:]

    # One-time b_dec init from real data
    if step == 0 and not ckpt_path.exists():
        sae.init_b_dec(x[:4096])

    # Dead feature mask: features that haven't fired for DEAD_TOKENS worth of steps
    dead_mask = None
    if tokens_seen >= DEAD_TOKENS:
        stale_steps = DEAD_TOKENS // BATCH_SIZE
        dead_mask = (sae._step - sae.last_fired) > stale_steps

    sae._step = step
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    x_hat, recon, aux, z = sae(x, dead_mask=dead_mask)
    loss = recon + ALPHA_AUX * aux

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    sae.renorm_decoder()

    tokens_seen += BATCH_SIZE
    step += 1
    pbar.update(BATCH_SIZE)

    if step % 25 == 0:
        with torch.no_grad():
            var = 1 - recon / x.var()
            l0 = (z != 0).float().sum(-1).mean().item()
            stale_steps = max(1, DEAD_TOKENS // BATCH_SIZE)
            dead_ct = ((sae._step - sae.last_fired) > stale_steps).sum().item()
        pbar.set_postfix(var_expl=f'{var.item():.3f}', L0=f'{l0:.1f}',
                         dead=dead_ct, lr=f'{lr_at(step):.1e}')

    # Drive checkpoint for idle-disconnect recovery
    if tokens_seen - last_ckpt_tok >= CKPT_EVERY_TOK:
        torch.save({'sae': sae.state_dict(), 'opt': opt.state_dict(),
                    'tokens_seen': tokens_seen, 'step': step}, ckpt_path)
        last_ckpt_tok = tokens_seen

pbar.close()
hook_handle.remove()
print(f'Training done in {(time.time()-t0)/60:.1f} min. Final var_expl ~ {var.item():.3f}')

# --- Save sae_lens-compatible safetensors + cfg.json, then push to HF ---
out_dir = Path(DRIVE_DIR) / 'sae_final'
out_dir.mkdir(exist_ok=True, parents=True)
state = {
    'W_enc': sae.W_enc.detach().cpu().contiguous(),
    'W_dec': sae.W_dec.detach().cpu().contiguous(),
    'b_enc': sae.b_enc.detach().cpu().contiguous(),
    'b_dec': sae.b_dec.detach().cpu().contiguous(),
}
save_file(state, str(out_dir / 'sae.safetensors'))
cfg = {
    'architecture': 'topk',
    'd_in': D_MODEL,
    'd_sae': N_FEATURES,
    'k': K_TOPK,
    'hook_name': f'blocks.{LAYER}.hook_resid_post',
    'model_name': MODEL_ID,
    'tokens_trained': tokens_seen,
}
(out_dir / 'cfg.json').write_text(json.dumps(cfg, indent=2))

api = HfApi()
create_repo(HF_REPO, exist_ok=True, private=False)
for f in ['sae.safetensors', 'cfg.json']:
    api.upload_file(path_or_fileobj=str(out_dir / f), path_in_repo=f,
                    repo_id=HF_REPO, repo_type='model')
print(f'Uploaded to https://huggingface.co/{HF_REPO}')

## Done. Your SAE is at `HF_REPO`

Quick sanity check: we'll re-enable the hook, pull a small held-out stream of activations, and measure how much variance the SAE explains plus which features light up most. Healthy numbers: **var_expl > 0.75** and **L0 ~= K_TOPK**.

In [ ]:
hook_handle = model.model.layers[LAYER].register_forward_hook(_hook)
sae.eval()

val_iter = activation_stream(token_packer(corpus))
buf_v = torch.empty(0, D_MODEL, device=DEVICE)
while buf_v.shape[0] < 5000:
    buf_v = torch.cat([buf_v, next(val_iter)], dim=0)
x_val = buf_v[:5000]

with torch.no_grad():
    x_hat, recon, _, z = sae(x_val)
    var_expl = (1 - recon / x_val.var()).item()
    l0 = (z != 0).float().sum(-1).mean().item()
    fire_counts = (z != 0).sum(0)
    top3 = fire_counts.topk(3).indices.tolist()

hook_handle.remove()
print(f'Held-out var_expl: {var_expl:.3f}   (target > 0.75)')
print(f'Held-out L0:       {l0:.1f}        (target ~ {K_TOPK})')
print(f'Top-3 most-active feature IDs: {top3}')
print(f'\nNext: try the Kaggle paper-grade notebook - wider SAE, longer run, full eval suite.')